# Municipal floodplain feasibility and provisional screening

Obtain the public 100 Year Flood Plain layer, verify complete retrieval and geometry/CRS, and screen the inventory. Capture dates are historical and model assumptions/coverage are undocumented. Results describe mapped overlap only. Nonintersection never becomes a low-hazard classification.

Run after notebook 01. Raw data and outputs stay local and Git-ignored. Existing snapshots are retained; reruns overwrite screening outputs and record a new run. Use a new snapshot ID/directory for updated source data.

In [ ]:
from pathlib import Path
import json

# In Colab, clone the repository and change into it before running this cell.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "config/ethekwini.yaml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run from the africa-map checkout or one of its subdirectories.")
CONFIG = json.loads((ROOT / "config/ethekwini.yaml").read_text())
print("Project:", CONFIG["project_id"])


In [ ]:
from urllib.request import urlopen
from urllib.parse import urlencode
from datetime import datetime, timezone
import ssl
import certifi

RAW = ROOT / CONFIG["floodplain_source_directory"]
RAW.mkdir(parents=True, exist_ok=True)
BASE = "https://gis.durban.gov.za/server/rest/services/WebViewers/Environmental_Services_Publisher/MapServer/9"
manifest_path = RAW / "download_manifest.json"
manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else []
ctx = ssl.create_default_context(cafile=certifi.where())
# The municipal server omitted its intermediate during acquisition. Obtain the
# issuer-signed certificate from its AIA URL if files must be fetched again.
# Root and hostname validation remain enabled; do not use an unverified context.
required = ["layer.json", "service.json", "count.json", "ids.json", "item.json", "native_sample.json"]
need_network = any(not (RAW / name).exists() for name in required)
if (RAW / "ids.json").exists():
    expected_batches = (len(json.loads((RAW / "ids.json").read_text())["objectIds"]) + 99) // 100
    need_network = need_network or any(not (RAW / f"features_{i:03d}.geojson").exists() for i in range(expected_batches))
if need_network:
    issuer_url = "http://crt.sectigo.com/SectigoPublicServerAuthenticationCADVR36.crt"
    with urlopen(issuer_url, timeout=30) as response:
        intermediate = ssl.DER_cert_to_PEM_cert(response.read())
    ctx.load_verify_locations(cadata=intermediate)
requests = {
    "layer.json": BASE + "?f=pjson",
    "service.json": BASE.rsplit("/", 1)[0] + "?f=pjson",
    "count.json": BASE + "/query?" + urlencode({"where": "1=1", "returnCountOnly": "true", "f": "json"}),
    "ids.json": BASE + "/query?" + urlencode({"where": "1=1", "returnIdsOnly": "true", "f": "json"}),
    "item.json": "https://gis.durban.gov.za/portal/sharing/rest/content/items/49cc2724d8d342a2924ef4bd3e3d394b?f=pjson",
    "native_sample.json": BASE + "/query?" + urlencode({"objectIds": "1", "outFields": "*", "returnGeometry": "true", "f": "json"}),
}
# Two phases: get IDs, then download exact ID batches to avoid truncation.
for phase in range(2):
    if phase == 1:
        ids = sorted(json.loads((RAW / "ids.json").read_text())["objectIds"])
        requests = {f"features_{i//100:03d}.geojson": BASE + "/query?" + urlencode({
            "objectIds": ",".join(map(str, ids[i:i+100])), "outFields": "*",
            "returnGeometry": "true", "outSR": "4326", "f": "geojson"}) for i in range(0, len(ids), 100)}
    for filename, url in requests.items():
        if (RAW / filename).exists():
            continue
        with urlopen(url, context=ctx, timeout=60) as response:
            content = response.read()
        payload = json.loads(content)
        if "error" in payload or payload.get("exceededTransferLimit"):
            raise RuntimeError("Failed or truncated query: " + filename)
        (RAW / filename).write_bytes(content)
        manifest.append({"file": filename, "url": url, "accessed_at_utc": datetime.now(timezone.utc).isoformat()})
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")
print("Source snapshot available locally; existing raw files retained.")


In [ ]:
from collections import Counter
import csv
import hashlib
import platform
import subprocess
import geopandas as gpd
import pandas as pd
import shapely
import pyproj
from pyproj import CRS
from heritage_flood.exposure import screen_floodplain

started = datetime.now(timezone.utc)
run_id = "floodplain_screen_" + started.strftime("%Y%m%dT%H%M%S%fZ")
layer = json.loads((RAW / "layer.json").read_text())
ids = json.loads((RAW / "ids.json").read_text())["objectIds"]
expected_count = json.loads((RAW / "count.json").read_text())["count"]
assert len(ids) == len(set(ids)) == expected_count
batches = [RAW / f"features_{i:03d}.geojson" for i in range((len(ids) + 99)//100)]
frames = []
for path in batches:
    payload = json.loads(path.read_text())
    assert not payload.get("exceededTransferLimit")
    frames.append(gpd.read_file(path))
flood = gpd.GeoDataFrame(pd.concat(frames, ignore_index=True), crs=frames[0].crs)
assert flood.crs.to_epsg() == 4326
assert len(flood) == expected_count and set(flood.OBJECTID) == set(ids)
assert flood.is_valid.all() and not flood.is_empty.any()
# Independently transform a native-coordinate sample and compare to outSR=4326.
native_path = RAW / "native_sample.json"
native = gpd.read_file(native_path)
native_payload = json.loads(native_path.read_text())
native_wkt = native_payload["spatialReference"].get("wkt") or layer["extent"]["spatialReference"]["wkt"]
native = native.set_crs(CRS.from_wkt(native_wkt), allow_override=True).to_crs(4326)
sample_id = int(native.OBJECTID.iloc[0])
sample_delta = native.geometry.iloc[0].hausdorff_distance(flood.loc[flood.OBJECTID == sample_id].geometry.iloc[0])
assert sample_delta < 1e-8

output = ROOT / CONFIG["output_directory"]
institutions_path = output / "institutions.geojson"
institutions = gpd.read_file(institutions_path)
# Check inventory bytes against its most recent recorded run.
with (ROOT / "metadata/processing_runs.csv").open(newline="") as handle:
    runs = list(csv.DictReader(handle))
inventory_run = next(r for r in reversed(runs) if r["run_id"] == institutions.run_id.iloc[0])
inventory_hashes = json.loads((ROOT / inventory_run["output_checksums_path"]).read_text())
assert hashlib.sha256(institutions_path.read_bytes()).hexdigest() == inventory_hashes[str(institutions_path.relative_to(ROOT))]
result = screen_floodplain(institutions, flood, analysis_crs=CONFIG["analysis_crs"],
                          buffer_distances_m=CONFIG["buffer_distances_m"])
result["inventory_run_id"] = result["run_id"]
result["run_id"] = run_id
result["floodplain_dataset_id"] = CONFIG["floodplain_dataset_id"]
result["return_period_label_years"] = 100
# Preserve contributing polygon IDs, river labels and source capture strings.
metric_flood = flood.to_crs(CONFIG["analysis_crs"])
metric_points = institutions.to_crs(CONFIG["analysis_crs"])
matching = [metric_flood.loc[metric_flood.intersects(point)] for point in metric_points.geometry]
result["intersecting_polygon_ids"] = [";".join(map(str, f.OBJECTID)) for f in matching]
result["intersecting_river_names"] = [";".join(sorted(set(f.RIVER_NAME.dropna().astype(str)))) for f in matching]
result["intersecting_capture_dates"] = [";".join(sorted(set(f.CAPT_DATE.dropna().astype(str)))) for f in matching]
result.to_file(output / "floodplain_screening.geojson", driver="GeoJSON")
result.drop(columns="geometry").to_csv(output / "floodplain_screening.csv", index=False)
result.to_crs(CONFIG["analysis_crs"]).to_file(output / "floodplain_screening.gpkg", layer="institutions", driver="GPKG")
metric_flood.to_file(output / "floodplain_screening.gpkg", layer="mapped_floodplain", driver="GPKG")
assert len(gpd.read_file(output / "floodplain_screening.gpkg", layer="institutions")) == len(institutions)
assert len(gpd.read_file(output / "floodplain_screening.gpkg", layer="mapped_floodplain")) == expected_count
summary = {
    "run_id": run_id, "floodplain_dataset_id": CONFIG["floodplain_dataset_id"],
    "polygon_count": len(flood), "valid_polygons": int(flood.is_valid.sum()),
    "bounds_wgs84": flood.total_bounds.tolist(), "source_crs_wkt": native_wkt,
    "native_sample_transform_difference_degrees": sample_delta,
    "capture_date_counts": dict(Counter(flood.CAPT_DATE.fillna("missing"))),
    "capture_date_interpretation": "Source capture strings, not verified model dates or effective dates",
    "institution_records": len(result), "point_intersections": int(result.point_intersects.sum()),
    "sensitivity_intersections": {str(r): int(result[f"buffer_{r}m_intersects"].sum()) for r in CONFIG["buffer_distances_m"]},
    "coverage_completeness": "not_established", "model_version": None,
    "climate_scenario": None, "modeled_depth_available": False,
    "redistribution_license": "No explicit license in inspected service/item metadata",
    "limitations": "Capture dates 1985–2012; model assumptions, baseline, completeness and positional accuracy undocumented. Nonintersection is not low hazard. Points independently unverified. No flood observation or damage inference.",
}
summary_path = ROOT / "metadata/ethekwini_floodplain_summary.json"
summary_path.write_text(json.dumps(summary, indent=2) + "\n")
print("Mapped point intersections:", summary["point_intersections"])
print("Sensitivity counts:", summary["sensitivity_intersections"])
print(result.loc[result.buffer_100m_intersects, ["institution_id", "name", "point_intersects", "distance_to_mapped_floodplain_m"]].to_string(index=False))


In [ ]:
run_directory = ROOT / "metadata/runs" / run_id
run_directory.mkdir(parents=True, exist_ok=False)
(run_directory / "config.json").write_text(json.dumps(CONFIG, indent=2) + "\n")
env = {"python": platform.python_version(), "geopandas": gpd.__version__, "pandas": pd.__version__,
       "shapely": shapely.__version__, "pyproj": pyproj.__version__}
(run_directory / "environment.json").write_text(json.dumps(env, indent=2) + "\n")
source_files = batches + [RAW / n for n in ["layer.json", "service.json", "count.json", "ids.json", "item.json", "native_sample.json"]]
inputs = source_files + [institutions_path, ROOT / CONFIG["boundary_path"]]
input_hashes = {str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest() for p in inputs}
(run_directory / "input_checksums.json").write_text(json.dumps(input_hashes, indent=2) + "\n")
source_hashes = {str(p.relative_to(RAW)): hashlib.sha256(p.read_bytes()).hexdigest() for p in source_files}
# Register the checksum of an input-file checksum manifest for the batched dataset.
source_manifest_path = ROOT / "metadata/ethekwini_floodplain_source_checksums.json"
source_manifest_path.write_text(json.dumps(source_hashes, indent=2) + "\n")
code = [ROOT / "src/heritage_flood/exposure.py", ROOT / "notebooks/03_floodplain_exposure.ipynb"]
(run_directory / "code_checksums.json").write_text(json.dumps({str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest() for p in code}, indent=2) + "\n")
outputs = [output / n for n in ["floodplain_screening.csv", "floodplain_screening.geojson", "floodplain_screening.gpkg"]] + [summary_path, source_manifest_path]
(run_directory / "checksums.json").write_text(json.dumps({str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest() for p in outputs}, indent=2) + "\n")
source_row = {"dataset_id": CONFIG["floodplain_dataset_id"], "title": "eThekwini 100 Year Flood Plain municipal layer 9",
    "provider": "eThekwini Municipality public GIS service", "version": "Snapshot retrieved 2026-10-20; model version unknown",
    "source_url_or_asset_id": BASE, "status": "accessed", "accessed_at_utc": manifest[0]["accessed_at_utc"],
    "geographic_coverage": "Mapped river polygons around eThekwini; completeness unknown",
    "resolution_or_scale": "Vector; scale and positional accuracy undocumented", "crs": "Native South Africa wg31 WKT; query output EPSG:4326",
    "license": "No explicit license in service or portal item metadata", "redistribution_conditions": "Keep downloaded geometry and screening outputs local pending clarification",
    "limitations": summary["limitations"] + " Source CAPT_DATE strings are not observation dates. The sha256 field hashes the retained source checksum manifest, not a single geometry file.",
    "storage_location": str(source_manifest_path.relative_to(ROOT)), "sha256": hashlib.sha256(source_manifest_path.read_bytes()).hexdigest()}
run_row = {"run_id": run_id, "started_at_utc": started.isoformat(),
    "input_dataset_ids": ";".join([CONFIG["floodplain_dataset_id"], CONFIG["institution_dataset_id"], CONFIG["boundary_dataset_id"]]),
    "code_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip(),
    "working_tree_dirty": str(bool(subprocess.check_output(["git", "status", "--porcelain"], cwd=ROOT, text=True).strip())).lower(),
    "config_snapshot_path": str((run_directory / "config.json").relative_to(ROOT)), "environment_path": str((run_directory / "environment.json").relative_to(ROOT)),
    "method": "Complete ID-batched municipal query; geometry/CRS checks; point intersects and 25/50/100 m distance sensitivity",
    "output_locations": ";".join(str(p.relative_to(ROOT)) for p in outputs), "output_checksums_path": str((run_directory / "checksums.json").relative_to(ROOT)),
    "validation_status": "partial", "notes": "Technical checks passed; scientific model validation and complete coverage unavailable. Polygon layer name supplies return-period label only. See research/floodplain_screening.md."}
for filename, key, row in [("datasets.csv", "dataset_id", source_row), ("processing_runs.csv", "run_id", run_row)]:
    path = ROOT / "metadata" / filename
    with path.open(newline="") as handle:
        reader = csv.DictReader(handle); fields, rows = reader.fieldnames, list(reader)
    previous = next((r for r in rows if r[key] == row[key]), None)
    if filename == "datasets.csv" and previous and previous["sha256"] != row["sha256"]:
        raise ValueError("Source changed: register a new snapshot ID.")
    with path.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields, lineterminator="\n")
        writer.writeheader(); writer.writerows([r for r in rows if r[key] != row[key]] + [row])
print("Registered run:", run_id)


## Interpretation

Read `research/floodplain_screening.md`. The 100-year label comes from the source layer name; model baseline, climate scenario, depth, coverage and accuracy are unknown. Buffers are location-sensitivity scenarios, not probabilities. No explicit redistribution license was found. A source metadata endpoint returned HTTP 404. Municipal service publication dates do not establish model currency. Do not use these results to rank investment or claim a site is safe.